<a href="https://colab.research.google.com/github/Kalito360/polyester-price-watcher./blob/main/Polyester_Price_Watcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Polyester Price Watcher (community tool)

An unofficial tool built by a community member using Polyester's official Python SDK. It reads public testnet market data and shows which markets moved the most. All prices are testnet paper prices and no login or API key is needed.

How to use it: run the first cell to install and then run the second cell to start the watcher. It refreshes every 30 seconds and stops by itself after 20 rounds.

Built by Web3Kalito (x.com/web3kalito1)

In [ ]:
!pip install -q "polyester-sdk==0.1.0a18"
!python --version

In [ ]:
import asyncio
from polyester import AsyncPolyester

POLL_SECONDS = 30
ROUNDS = 20
TOP_N = 5
PRICE_DIVISOR = 1_000_000_000   # raw price units, checked against the testnet site


def price_of(m):
    if m.last_price is None:
        return None
    return m.last_price.ticks / PRICE_DIVISOR


def change_pct(m):
    try:
        return float(m.change_24h_bp) / 100
    except (TypeError, ValueError):
        return None


async def load_names(client):
    r = client.wait_for_catalogs()
    if hasattr(r, "__await__"):
        await r
    cfg = client.catalogs.spot_config
    if callable(cfg):
        cfg = cfg()
        if hasattr(cfg, "__await__"):
            cfg = await cfg
    return {p["symbolId"]: p["symbol"] for p in cfg["pairs"]}


async def snapshot(client, names):
    overview = await client.market_overview.list(limit=100)
    rows = {}
    for m in overview.markets:
        p = price_of(m)
        if p:
            name = names.get(m.symbol_id, f"market-{m.symbol_id}")
            rows[m.symbol_id] = (name, p, change_pct(m))
    return rows


async def main():
    async with AsyncPolyester() as client:
        names = await load_names(client)
        start = await snapshot(client, names)

        print(f"Watching {len(start)} markets. Baseline saved.")
        print("\nPRICE CHECK (compare with the Polyester website):")
        for name, p, ch in start.values():
            print(f"  {name:<12} {p:>14.4f}")

        for i in range(1, ROUNDS + 1):
            await asyncio.sleep(POLL_SECONDS)
            now = await snapshot(client, names)

            moves = []
            for sid, (name, p, ch) in now.items():
                if sid in start:
                    base = start[sid][1]
                    moves.append((name, (p - base) / base * 100, p, ch))
            moves.sort(key=lambda x: abs(x[1]), reverse=True)

            print(f"\nRound {i}/{ROUNDS}   name | price | since start | 24h")
            for name, pct, p, ch in moves[:TOP_N]:
                day = f"{ch:+.2f}%" if ch is not None else "n/a"
                print(f"{name:<12} {p:>14.4f}  {pct:+.2f}%  {day}")


await main()